In [3]:
import pandas as pd 
import numpy as np

In [4]:
customers_df = pd.read_csv('customers.csv')
calls_df = pd.read_csv('calls.csv')
reason_df = pd.read_csv('reason.csv')
sentiment_df = pd.read_csv('sentiment_statistics.csv')

In [5]:
print(f"Customers: {customers_df.shape}")
print(f"Calls: {calls_df.shape}")
print(f"Reasons: {reason_df.shape}")
print(f"Sentiment: {sentiment_df.shape}")

Customers: (71810, 3)
Calls: (71810, 7)
Reasons: (66653, 2)
Sentiment: (71810, 6)


In [6]:
# Convert columns to Datetime type
calls_df['call_start_datetime']=pd.to_datetime(calls_df['call_start_datetime'])
calls_df['agent_assigned_datetime']=pd.to_datetime(calls_df['agent_assigned_datetime'])
calls_df['call_end_datetime']=pd.to_datetime(calls_df['call_end_datetime'])

In [7]:
calls_df['wait_time_seconds']= (calls_df['agent_assigned_datetime']-calls_df['call_start_datetime']).dt.total_seconds()
calls_df['talk_duration_seconds']= (calls_df['call_end_datetime']-calls_df['agent_assigned_datetime']).dt.total_seconds()
print(calls_df[['wait_time_seconds','talk_duration_seconds']].head())

   wait_time_seconds  talk_duration_seconds
0              420.0                 1860.0
1              180.0                  720.0
2              480.0                 1140.0
3              300.0                  420.0
4              600.0                  540.0


In [8]:
customers_df.duplicated().sum()

np.int64(0)

In [9]:
calls_df.duplicated().sum()

np.int64(0)

In [10]:
reason_df.duplicated().sum()

np.int64(0)

In [11]:
sentiment_df.duplicated().sum()

np.int64(0)

In [12]:
customers_df.isnull().sum()

customer_id             0
customer_name           0
elite_level_code    25767
dtype: int64

In [13]:
calls_df.isnull().sum()

call_id                    0
customer_id                0
agent_id                   0
call_start_datetime        0
agent_assigned_datetime    0
call_end_datetime          0
call_transcript            0
wait_time_seconds          0
talk_duration_seconds      0
dtype: int64

In [14]:
reason_df.isnull().sum()

call_id                0
primary_call_reason    0
dtype: int64

In [15]:
sentiment_df.isnull().sum()

call_id                      0
agent_id                     0
agent_tone                 217
customer_tone                0
average_sentiment          109
silence_percent_average      0
dtype: int64

In [16]:
# Process the customer category column
customers_df['elite_level_code']= customers_df['elite_level_code'].fillna('Standard')

In [17]:
# Addressing missing employee tone
sentiment_df['agent_tone'] = sentiment_df['agent_tone'].fillna('Unknown')

In [18]:
#Process the missing average sentiment (let it neutral = 0)
sentiment_df['average_sentiment'] = sentiment_df['average_sentiment'].fillna(0)

In [19]:
print("Nulls left in Customers:", customers_df['elite_level_code'].isnull().sum())
print("Nulls left in Sentiment:", sentiment_df['agent_tone'].isnull().sum())

Nulls left in Customers: 0
Nulls left in Sentiment: 0


In [20]:
master_df = pd.merge(calls_df,customers_df, on='customer_id', how='left')
master_df=pd.merge(master_df,reason_df, on='call_id', how='left')
master_df=pd.merge(master_df,sentiment_df,on = ['call_id','agent_id'] , how='left')
print(master_df.shape)
print(master_df.head(5))

(71810, 16)
      call_id  customer_id  agent_id call_start_datetime  \
0  4667960400   2033123310    963118 2024-07-31 23:56:00   
1  1122072124   8186702651    519057 2024-08-01 00:03:00   
2  6834291559   2416856629    158319 2024-07-31 23:59:00   
3  2266439882   1154544516    488324 2024-08-01 00:05:00   
4  1211603231   5214456437    721730 2024-08-01 00:04:00   

  agent_assigned_datetime   call_end_datetime  \
0     2024-08-01 00:03:00 2024-08-01 00:34:00   
1     2024-08-01 00:06:00 2024-08-01 00:18:00   
2     2024-08-01 00:07:00 2024-08-01 00:26:00   
3     2024-08-01 00:10:00 2024-08-01 00:17:00   
4     2024-08-01 00:14:00 2024-08-01 00:23:00   

                                     call_transcript  wait_time_seconds  \
0  \n\nAgent: Thank you for calling United Airlin...              420.0   
1  \n\nAgent: Thank you for calling United Airlin...              180.0   
2  \n\nAgent: Thank you for calling United Airlin...              480.0   
3  \n\nAgent: Thank you for call

In [21]:
elite_mapping= {
    '0.0': 'Standard',
    '1.0': 'Bronze Elite',
    '2.0': 'Silver Elite',
    '3.0': 'Gold Elite',
    '4.0': 'Platinum Elite',
    '5.0': 'Diamond Elite',
    'Standard': 'Standard'
}

master_df['elite_level_code'] = master_df['elite_level_code'].astype(str).str.strip()
master_df['elite_level_code'] = master_df['elite_level_code'].map(elite_mapping).fillna('Standard')
print(master_df['elite_level_code'].value_counts())

elite_level_code
Standard          40154
Bronze Elite      14370
Silver Elite       8028
Gold Elite         5736
Platinum Elite     2125
Diamond Elite      1397
Name: count, dtype: int64


In [22]:
master_df['elite_level_code'].head(10)

0    Platinum Elite
1          Standard
2          Standard
3      Silver Elite
4          Standard
5     Diamond Elite
6          Standard
7      Silver Elite
8      Bronze Elite
9          Standard
Name: elite_level_code, dtype: object

In [23]:
pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [24]:
import duckdb
sentiment_query="""
select 
agent_tone,
count(call_id) as handeled_angry_calls,
Round(avg(talk_duration_seconds)/60,2) as avg_talk_duration_minutes,
Round(avg(average_sentiment),3) as final_avg_sentiment,
Round(aVg(silence_percent_average)*100,1) as avg_silence_percentage
from master_df
where customer_tone='angry'
group by agent_tone
order by final_avg_sentiment desc;
"""
sentiment_insight = duckdb.query(sentiment_query).to_df()
sentiment_insight

,agent_tone,handeled_angry_calls,avg_talk_duration_minutes,final_avg_sentiment,avg_silence_percentage
0,polite,18,5.67,0.676,33.4
1,calm,4922,10.27,0.104,29.5
2,neutral,8618,12.51,-0.080,28.4
3,frustrated,732,10.51,-0.328,25.7
4,Unknown,41,4.88,-0.332,26.2
5,angry,77,8.30,-0.595,23.5


In [25]:
sla_query="""
select 
elite_level_code ,
count(call_id) as total_calls,
round(avg(wait_time_seconds),2) as avg_wait_time_sec,
round(max(wait_time_seconds)) as max_wait_time_seconds
from master_df
group by elite_level_code
order by avg_wait_time_sec asc
"""
sla_insight=duckdb.query(sla_query).to_df()
sla_insight

,elite_level_code,total_calls,avg_wait_time_sec,max_wait_time_seconds
0,Diamond Elite,1397,411.41,900.0
1,Platinum Elite,2125,413.31,900.0
2,Gold Elite,5736,429.34,900.0
3,Silver Elite,8028,432.77,900.0
4,Bronze Elite,14370,436.92,900.0
5,Standard,40154,441.23,900.0


In [26]:
!pip install sqlalchemy pymysql
from sqlalchemy import create_engine
username = 'root'
password = 'your_password' 
host = '127.0.0.1'
port = '3306'
database = 'call_center_db'

engine = create_engine(f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}')
master_df.to_sql('master_calls', engine, if_exists='replace', index=False)

71810